# Notebook 5: Export to TFLite (INT8 quantized, mobile-ready)

Requires Notebook 4's 7 `student_*.pt` checkpoints on Drive. Converts each PyTorch MobileNetV3 student into an INT8-quantized `.tflite` file: PyTorch -> ONNX -> TensorFlow SavedModel (`onnx2tf`) -> TFLite (`tf.lite.TFLiteConverter`, INT8 post-training quantization, calibrated on each target's own held-out validation images).

**Design choice**: each exported model takes raw 0-255 pixel input and does ImageNet normalization *inside* the graph (via a small wrapper module before export). This means the mobile app just feeds it a raw decoded image — no need to replicate the float normalization math app-side, which removes a whole class of train/deploy preprocessing-mismatch bugs.

**Heads up**: the `onnx2tf` conversion step is the part of this pipeline most likely to need live debugging against whatever library versions the runtime actually has — expect possible iteration here, same as LeafNet's schema surprise in Notebook 2.

## Task 1: Setup — install conversion toolchain, load data + student checkpoints

In [ ]:
!pip install -q timm pillow pandas scikit-learn onnx onnxsim onnx_graphsurgeon sng4onnx onnx2tf

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import zipfile
import pandas as pd

DRIVE_ZIP = "/content/drive/MyDrive/crop_disease/data.zip"
UNIFIED_ROOT = "/content/data"
STAGE_DIR = "/content/drive/MyDrive/crop_disease"
EXPORT_DIR = os.path.join(STAGE_DIR, "tflite_export")
os.makedirs(EXPORT_DIR, exist_ok=True)

assert os.path.exists(DRIVE_ZIP), f"{DRIVE_ZIP} not found — run Notebook 1 first"
os.makedirs(UNIFIED_ROOT, exist_ok=True)
with zipfile.ZipFile(DRIVE_ZIP) as zf:
    zf.extractall(UNIFIED_ROOT)

manifest = pd.read_csv(os.path.join(UNIFIED_ROOT, "manifest.csv"))
CROPS = sorted(manifest["crop"].unique())

STUDENT_CKPT_PATHS = {"stage1_crop": os.path.join(STAGE_DIR, "student_stage1_crop_classifier.pt")}
STUDENT_CKPT_PATHS.update({
    f"stage2_{c}": os.path.join(STAGE_DIR, f"student_stage2_{c}_disease.pt") for c in CROPS
})
for name, p in STUDENT_CKPT_PATHS.items():
    assert os.path.exists(p), f"{p} not found — run Notebook 4 first ({name})"

print(manifest.shape)
print("Targets:", list(STUDENT_CKPT_PATHS))

In [ ]:
import torch
import tensorflow as tf
print("Torch CUDA available:", torch.cuda.is_available())
print("TensorFlow version:", tf.__version__)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Task 2: Shared utilities — group-aware val split (for calibration data), preprocessing wrapper

In [ ]:
import re
import timm
import torch.nn as nn
from sklearn.model_selection import GroupShuffleSplit
from PIL import Image
import numpy as np

IMG_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# Same heuristic as Notebooks 3-4: recover a shared "source photo" key so
# calibration images come from the genuinely held-out validation split, not
# from images the model was trained on.
AUG_PREFIX_RE = re.compile(
    r"^(?:[a-z0-9]+_primary_|multi_crop_supplement_)"
    r"(?:resized_|rotated_|zoomed_|cropped_|flipped_horiz_|flipped_vert_|flipped_)*",
    re.IGNORECASE,
)

def build_group_key(filepath, label):
    stem = os.path.splitext(os.path.basename(filepath))[0]
    stripped = AUG_PREFIX_RE.sub("", stem, count=1)
    return f"{label}::{stripped.lower()}"

def get_val_split(df, label_col):
    df = df.copy()
    df["group_key"] = [build_group_key(fp, lbl) for fp, lbl in zip(df["filepath"], df[label_col])]
    gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
    _, val_idx = next(gss.split(df, groups=df["group_key"]))
    return df.iloc[val_idx]

class PreprocessWrapper(nn.Module):
    """Wraps a trained model so the exported graph accepts raw 0-255 pixel
    input (NCHW) and does ImageNet normalization internally, so the mobile
    app never needs to replicate that math."""
    def __init__(self, model, mean, std):
        super().__init__()
        self.model = model
        self.register_buffer("mean", torch.tensor(mean).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor(std).view(1, 3, 1, 1))

    def forward(self, x):
        x = x / 255.0
        x = (x - self.mean) / self.std
        return self.model(x)

## Task 3: PyTorch -> ONNX export

In [ ]:
def export_to_onnx(target_name, ckpt_path, onnx_path):
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    classes = ckpt["classes"]
    model = timm.create_model("mobilenetv3_large_100", pretrained=False, num_classes=len(classes))
    model.load_state_dict(ckpt["model_state"])
    model.eval()

    wrapped = PreprocessWrapper(model, IMAGENET_MEAN, IMAGENET_STD).eval()
    dummy_input = torch.randn(1, 3, IMG_SIZE, IMG_SIZE) * 255.0

    torch.onnx.export(
        wrapped,
        dummy_input,
        onnx_path,
        input_names=["pixel_input"],
        output_names=["logits"],
        opset_version=17,
        dynamic_axes=None,  # fixed batch size 1 — simplest, most reliable for mobile export
    )
    print(f"  [{target_name}] exported ONNX -> {onnx_path} ({len(classes)} classes)")
    return classes

## Task 4: ONNX -> TFLite INT8 quantized conversion

In [ ]:
import onnx2tf

def convert_to_tflite_int8(target_name, onnx_path, saved_model_dir, tflite_path, val_df):
    onnx2tf.convert(
        input_onnx_file_path=onnx_path,
        output_folder_path=saved_model_dir,
        copy_onnx_input_output_names_to_tflite=True,
        non_verbose=True,
    )

    loaded = tf.saved_model.load(saved_model_dir)
    input_spec = list(loaded.signatures["serving_default"].structured_input_signature[1].values())[0]
    input_shape = input_spec.shape.as_list()
    channels_first = len(input_shape) == 4 and input_shape[1] == 3
    print(f"  [{target_name}] converted SavedModel input shape: {input_shape} ({'NCHW' if channels_first else 'NHWC'})")

    val_paths = val_df["filepath"].tolist()[:200]  # calibration sample, held-out images only

    def representative_dataset_gen():
        for fp in val_paths:
            img = Image.open(fp).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
            arr = np.array(img, dtype=np.float32)  # HWC, 0-255
            if channels_first:
                arr = np.transpose(arr, (2, 0, 1))  # CHW
            arr = np.expand_dims(arr, axis=0)
            yield [arr]

    converter = tf.lite.TFLiteConverter.from_saved_model(saved_model_dir)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset = representative_dataset_gen
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type = tf.uint8
    converter.inference_output_type = tf.float32
    tflite_model = converter.convert()

    with open(tflite_path, "wb") as f:
        f.write(tflite_model)

    size_mb = os.path.getsize(tflite_path) / 1e6
    print(f"  [{target_name}] TFLite INT8 saved -> {tflite_path} ({size_mb:.2f} MB)")
    return channels_first

## Task 5: Run export + quantize for all 7 targets (resumable)

In [ ]:
EXPORT_META = {}

targets = [("stage1_crop", manifest, "crop")]
targets += [(f"stage2_{c}", manifest[manifest["crop"] == c], "disease") for c in CROPS]

for target_name, df, label_col in targets:
    print(f"\n=== {target_name} ===")
    tflite_path = os.path.join(EXPORT_DIR, f"{target_name}.tflite")
    if os.path.exists(tflite_path):
        print(f"  [{target_name}] {tflite_path} already exists, skipping export (delete the file to force re-export)")
        continue

    onnx_path = os.path.join(EXPORT_DIR, f"{target_name}.onnx")
    saved_model_dir = os.path.join(EXPORT_DIR, f"{target_name}_saved_model")
    val_df = get_val_split(df, label_col)

    classes = export_to_onnx(target_name, STUDENT_CKPT_PATHS[target_name], onnx_path)
    channels_first = convert_to_tflite_int8(target_name, onnx_path, saved_model_dir, tflite_path, val_df)
    EXPORT_META[target_name] = {"classes": classes, "channels_first": channels_first}

If this cell stops partway, just rerun it — it skips any target whose `.tflite` file already exists in `tflite_export/` on Drive.

## Task 6: Quantized accuracy check — TFLite vs original PyTorch student

In [ ]:
def evaluate_tflite(tflite_path, val_df, label_col, classes, channels_first):
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]
    label_to_idx = {c: i for i, c in enumerate(classes)}

    correct, total = 0, 0
    for _, row in val_df.iterrows():
        img = Image.open(row["filepath"]).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
        arr = np.array(img, dtype=np.uint8)
        if channels_first:
            arr = np.transpose(arr, (2, 0, 1))
        arr = np.expand_dims(arr, axis=0)
        interpreter.set_tensor(input_details["index"], arr)
        interpreter.invoke()
        pred = np.argmax(interpreter.get_tensor(output_details["index"])[0])
        correct += int(pred == label_to_idx[row[label_col]])
        total += 1
    return correct / total if total else 0.0

In [ ]:
rows = []
for target_name, df, label_col in targets:
    tflite_path = os.path.join(EXPORT_DIR, f"{target_name}.tflite")
    student_ckpt = torch.load(STUDENT_CKPT_PATHS[target_name], map_location=DEVICE)
    classes = student_ckpt["classes"]
    channels_first = EXPORT_META.get(target_name, {}).get("channels_first")
    if channels_first is None:
        # Recover this if resuming from a prior session where EXPORT_META wasn't populated
        loaded = tf.saved_model.load(os.path.join(EXPORT_DIR, f"{target_name}_saved_model"))
        input_spec = list(loaded.signatures["serving_default"].structured_input_signature[1].values())[0]
        shape = input_spec.shape.as_list()
        channels_first = len(shape) == 4 and shape[1] == 3

    val_df = get_val_split(df, label_col)
    tflite_acc = evaluate_tflite(tflite_path, val_df, label_col, classes, channels_first)
    size_mb = os.path.getsize(tflite_path) / 1e6

    rows.append({
        "target": target_name,
        "num_classes": len(classes),
        "pytorch_student_val_acc": student_ckpt["val_acc"],
        "tflite_int8_val_acc": tflite_acc,
        "tflite_size_MB": round(size_mb, 2),
    })

summary = pd.DataFrame(rows)
summary["quant_acc_delta"] = summary["tflite_int8_val_acc"] - summary["pytorch_student_val_acc"]
print(summary.to_string(index=False))

Watch `quant_acc_delta` — a large negative number means INT8 quantization hurt real accuracy meaningfully, which would call for QAT (quantization-aware training) instead of post-training quantization. This is a Python-side proxy check only; the spec's real requirement (test on an actual mid-range Android phone) still needs to happen before shipping.

## Task 7: On-device footprint check

In [ ]:
stage1_size = summary.loc[summary["target"] == "stage1_crop", "tflite_size_MB"].iloc[0]
stage2_sizes = summary.loc[summary["target"] != "stage1_crop", "tflite_size_MB"]

print(f"Stage-1 (crop classifier): {stage1_size:.2f} MB")
print(f"Stage-2 heads (per crop): min={stage2_sizes.min():.2f} MB, max={stage2_sizes.max():.2f} MB, avg={stage2_sizes.mean():.2f} MB")

worst_case_total = stage1_size + stage2_sizes.max()
print(f"\nWorst-case on-device footprint (Stage-1 + one active Stage-2 head): {worst_case_total:.2f} MB")
print("Target from the design spec: <15 MB" + (" — MET" if worst_case_total < 15 else " — OVER, consider a smaller student or more aggressive quantization"))